# Nonlinear rigid body rotation: Euler's equations and the tennis racket theorem

Everything in [`dgs/gyroscopes.py`](../dgs/gyroscopes.py) up to this point
— precession, nutation, the small-angle pendulum — lives in the *linear*
regime: a fast-spinning, nearly-symmetric top under a small perturbing
torque. A genuinely **nonlinear** rigid body — three distinct principal
moments of inertia $I_1<I_2<I_3$, no external torque — obeys Euler's
equations:

$$I_1\dot\omega_1=(I_2-I_3)\omega_2\omega_3, \quad
I_2\dot\omega_2=(I_3-I_1)\omega_3\omega_1, \quad
I_3\dot\omega_3=(I_1-I_2)\omega_1\omega_2$$

The famous, genuinely surprising result: spinning about the axis of
**largest** or **smallest** moment of inertia is stable, but spinning about
the **intermediate** axis is unstable — the "tennis racket theorem" (a
tossed racket flips over mid-air if spun about its middle axis), also known
as the Dzhanibekov effect after the cosmonaut who filmed a spinning wingnut
doing exactly this in zero gravity.


In [1]:
import sys, pathlib
import numpy as np
import matplotlib.pyplot as plt

REPO = pathlib.Path(r"D:/Summer2026/Dispersion-Assisted-GS-Phase-Recovery")
sys.path.insert(0, str(REPO))
from dgs.gyroscopes import (
    integrate_euler_rigid_body, rotational_energy_and_momentum,
    intermediate_axis_stability_coefficients, tennis_racket_theorem_demo,
)

checks = []


def check(label, condition):
    checks.append((label, bool(condition)))
    print(f"{'PASS' if condition else 'FAIL'}  —  {label}")


## 1. The integrator is actually correct: energy and angular momentum are conserved

For torque-free rotation, both $T=\tfrac12(I_1\omega_1^2+I_2\omega_2^2+I_3\omega_3^2)$
and $|\mathbf L|^2=(I_1\omega_1)^2+(I_2\omega_2)^2+(I_3\omega_3)^2$ are
exact constants of motion — a real correctness check on the RK4 integrator,
not just a plausibility check on the physics.


In [2]:
I1, I2, I3 = 1.0, 2.0, 3.0
run = integrate_euler_rigid_body([0.1, 3.0, 0.2], I1, I2, I3, t_max=10.0, dt=0.0005)
cons = rotational_energy_and_momentum(run["omega"], I1, I2, I3)
T, L2 = cons["T"], cons["L_squared"]

T_rel_var = (T.max() - T.min()) / T.mean()
L2_rel_var = (L2.max() - L2.min()) / L2.mean()
print(f"Kinetic energy T: relative variation over 10s = {T_rel_var:.2e}")
print(f"|L|^2:            relative variation over 10s = {L2_rel_var:.2e}")

check("Kinetic energy conserved to machine precision", T_rel_var < 1e-8)
check("|L|^2 conserved to machine precision", L2_rel_var < 1e-8)


Kinetic energy T: relative variation over 10s = 1.92e-14
|L|^2:            relative variation over 10s = 1.95e-14
PASS  —  Kinetic energy conserved to machine precision
PASS  —  |L|^2 conserved to machine precision


## 2. The analytic prediction: linearize about each principal axis

Perturbing slightly off pure rotation about axis $k$ turns Euler's
equations into a linear 2nd-order ODE for the two transverse components,
with a coefficient whose *sign* determines stability — negative means
oscillatory (stable), positive means exponential growth (unstable).


In [3]:
stab = intermediate_axis_stability_coefficients(I1, I2, I3)
for axis, info in stab.items():
    verdict = "stable (oscillatory)" if info["stable"] else "UNSTABLE (exponential growth)"
    print(f"{axis}: coefficient = {info['value']:+.4f}   -> {verdict}")
    print(f"    symbolic form: {info['symbolic']}")

check("Axis 1 (smallest I) predicted stable", stab["axis1"]["stable"])
check("Axis 2 (intermediate I) predicted UNSTABLE", not stab["axis2"]["stable"])
check("Axis 3 (largest I) predicted stable", stab["axis3"]["stable"])


axis1: coefficient = -0.3333   -> stable (oscillatory)
    symbolic form: -(I1 - I2)*(I1 - I3)/(I2*I3)
axis2: coefficient = +0.3333   -> UNSTABLE (exponential growth)
    symbolic form: (I1 - I2)*(I2 - I3)/(I1*I3)
axis3: coefficient = -1.0000   -> stable (oscillatory)
    symbolic form: -(I1 - I3)*(I2 - I3)/(I1*I2)
PASS  —  Axis 1 (smallest I) predicted stable
PASS  —  Axis 2 (intermediate I) predicted UNSTABLE
PASS  —  Axis 3 (largest I) predicted stable


## 3. The numeric demonstration: does a tiny wobble actually grow into a flip?

Spin the body almost exactly about each principal axis in turn (a $10^{-3}$
perturbation in the other two components) and watch what happens to those
transverse components over 20 seconds.


In [4]:
demo = tennis_racket_theorem_demo(I1, I2, I3, Omega=5.0, eps=1e-3, t_max=20.0, dt=0.001)
for axis, info in demo.items():
    print(f"{axis}: transverse component grew from 1e-3 to {info['max_transverse']:.4f}   "
          f"({'FLIPPED' if info['flipped'] else 'stayed bounded'})")

check("Spin about axis 1 stays bounded (matches analytic prediction)", not demo["axis1"]["flipped"])
check("Spin about axis 2 actually flips (matches analytic prediction)", demo["axis2"]["flipped"])
check("Spin about axis 3 stays bounded (matches analytic prediction)", not demo["axis3"]["flipped"])


axis1: transverse component grew from 1e-3 to 0.0020   (stayed bounded)
axis2: transverse component grew from 1e-3 to 5.0000   (FLIPPED)
axis3: transverse component grew from 1e-3 to 0.0014   (stayed bounded)
PASS  —  Spin about axis 1 stays bounded (matches analytic prediction)
PASS  —  Spin about axis 2 actually flips (matches analytic prediction)
PASS  —  Spin about axis 3 stays bounded (matches analytic prediction)


## 4. Watching it happen

Plot all three $\omega$ components for each of the three spin cases — the
intermediate-axis case should look qualitatively different: not a small
wobble, but full-amplitude oscillation between all three axes.


In [5]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
titles = {"axis1": "Spin about axis 1 (smallest I) — stable",
          "axis2": "Spin about axis 2 (intermediate I) — UNSTABLE",
          "axis3": "Spin about axis 3 (largest I) — stable"}
initial_conditions = {"axis1": [5.0, 1e-3, 1e-3], "axis2": [1e-3, 5.0, 1e-3], "axis3": [1e-3, 1e-3, 5.0]}

for ax, (axis_name, omega0) in zip(axes, initial_conditions.items()):
    run = integrate_euler_rigid_body(omega0, I1, I2, I3, t_max=20.0, dt=0.005)
    ax.plot(run["t"], run["omega"][:, 0], label=r"$\omega_1$")
    ax.plot(run["t"], run["omega"][:, 1], label=r"$\omega_2$")
    ax.plot(run["t"], run["omega"][:, 2], label=r"$\omega_3$")
    ax.set_title(titles[axis_name], fontsize=10)
    ax.set_xlabel("t (s)")
    ax.legend(fontsize=8)
axes[0].set_ylabel(r"$\omega$ (rad/s)")
plt.tight_layout()
out_png = str(REPO / "notebooks" / "tennis_racket_theorem_flip.png")
plt.savefig(out_png, dpi=130)
plt.close(fig)
print("saved", out_png)


saved D:\Summer2026\Dispersion-Assisted-GS-Phase-Recovery\notebooks\tennis_racket_theorem_flip.png


The intermediate-axis case doesn't just wobble a little more — the
transverse components grow all the way up to the spin magnitude itself,
because the body is genuinely tumbling: periodically dumping its rotation
into the other two axes and back, the same "flip" a thrown tennis racket or
Dzhanibekov's wingnut visibly does.

## Final grade

In [6]:
failures = [label for label, ok in checks if not ok]
print(f"{len(checks) - len(failures)}/{len(checks)} checks passed")

if failures:
    raise AssertionError("Failed checks: " + ", ".join(failures))
else:
    print("\nALL CHECKS PASSED — Euler's equations conserve energy and angular momentum "
          "exactly, the analytic stability prediction matches the numeric simulation for "
          "all three principal axes, and the intermediate-axis instability (tennis racket "
          "theorem) actually produces a full flip, not just a bigger wobble.")


8/8 checks passed

ALL CHECKS PASSED — Euler's equations conserve energy and angular momentum exactly, the analytic stability prediction matches the numeric simulation for all three principal axes, and the intermediate-axis instability (tennis racket theorem) actually produces a full flip, not just a bigger wobble.
